In [68]:
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

RAW = Path('raw')
CLEAN = Path('cleaned')
CLEAN.mkdir(exist_ok=True)

products = pd.read_csv(RAW / 'olist_products_dataset.csv')

In [69]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [70]:
products.isna().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [71]:
products['product_id'].is_unique

True

In [72]:
products['product_category_name'].value_counts()

product_category_name
cama_mesa_banho                  3029
esporte_lazer                    2867
moveis_decoracao                 2657
beleza_saude                     2444
utilidades_domesticas            2335
                                 ... 
fashion_roupa_infanto_juvenil       5
casa_conforto_2                     5
pc_gamer                            3
seguros_e_servicos                  2
cds_dvds_musicais                   1
Name: count, Length: 73, dtype: int64

In [73]:
products.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


In [74]:
cat_missing = products['product_category_name'].isna()

products.loc[cat_missing,['product_name_lenght','product_description_lenght','product_photos_qty']].isna().all().all()

np.True_

In [75]:
products[products['product_weight_g'].isna()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [76]:
products[['product_weight_g','product_length_cm','product_height_cm','product_width_cm']].describe()

,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32949.000000,32949.000000,32949.000000,32949.000000
mean,2276.472488,30.815078,16.937661,23.196728
std,4282.038731,16.914458,13.637554,12.079047
min,0.000000,7.000000,2.000000,6.000000
25%,300.000000,18.000000,8.000000,15.000000
50%,700.000000,25.000000,13.000000,20.000000
75%,1900.000000,38.000000,21.000000,30.000000
max,40425.000000,105.000000,105.000000,118.000000


In [77]:
(products['product_weight_g'] == 0).sum()

np.int64(4)

In [78]:
trans = pd.read_csv(RAW/ 'product_category_name_translation.csv')
trans.info()

<class 'pandas.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          71 non-null     str  
 1   product_category_name_english  71 non-null     str  
dtypes: str(2)
memory usage: 1.2 KB


In [79]:
trans['product_category_name'].is_unique

True

In [80]:
set(products['product_category_name'].dropna()) - set(trans['product_category_name'].dropna())

{'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}

In [81]:
items = pd.read_csv(CLEAN/'order_items_clean.csv', usecols=['order_id','product_id'])

In [82]:
(~products['product_id'].isin(items['product_id'])).sum()

np.int64(0)

In [83]:
(~items['product_id'].isin(products['product_id'])).sum()

np.int64(0)

In [84]:
print(products[products['product_weight_g'] == 0])
print((products[['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']].dropna() % 1 != 0).sum())
cat = products['product_category_name'].dropna()
print((cat != cat.str.strip()).sum(), (cat != cat.str.lower()).sum())

                             product_id product_category_name  \
9769   81781c0fed9fe1ad6e8c81fca1e1cb08       cama_mesa_banho   
13683  8038040ee2a71048d4bdbbdc985b69ab       cama_mesa_banho   
14997  36ba42dd187055e1fbe943b2d11430ca       cama_mesa_banho   
32079  e673e90efa65a5409ff4196c038bb5af       cama_mesa_banho   

       product_name_lenght  product_description_lenght  product_photos_qty  \
9769                  51.0                       529.0                 1.0   
13683                 48.0                       528.0                 1.0   
14997                 53.0                       528.0                 1.0   
32079                 53.0                       528.0                 1.0   

       product_weight_g  product_length_cm  product_height_cm  \
9769                0.0               30.0               25.0   
13683               0.0               30.0               25.0   
14997               0.0               30.0               25.0   
32079               0.0

In [85]:
# 1. Fix the column names
products = products.rename(columns={
    'product_name_lenght': 'product_name_length',
    'product_description_lenght': 'product_description_length',
})

In [86]:
# 2. Whole numbers, keeping the empty cells
int_cols = ['product_name_length', 'product_description_length', 'product_photos_qty',
            'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
products[int_cols] = products[int_cols].astype('Int64')
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   product_id                  32951 non-null  str  
 1   product_category_name       32341 non-null  str  
 2   product_name_length         32341 non-null  Int64
 3   product_description_length  32341 non-null  Int64
 4   product_photos_qty          32341 non-null  Int64
 5   product_weight_g            32949 non-null  Int64
 6   product_length_cm           32949 non-null  Int64
 7   product_height_cm           32949 non-null  Int64
 8   product_width_cm            32949 non-null  Int64
dtypes: Int64(7), str(2)
memory usage: 2.5 MB


In [87]:
# 3. English category names (adding the 2 missing translations by hand)
extra = pd.DataFrame({
    'product_category_name': ['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'],
    'product_category_name_english': ['pc_gamer', 'portable_kitchen_and_food_preparers'],
})
trans_full = pd.concat([trans, extra], ignore_index=True)
products = products.merge(trans_full, on='product_category_name', how='left')
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')
print(len(products))                                         # still 32,951

32951


In [88]:
# 4. Flag the weight problems
products['has_size_issue'] = products['product_weight_g'].isna() | (products['product_weight_g'] == 0)
print(products['has_size_issue'].value_counts())             # True 6 / False 32,945

has_size_issue
False    32945
True         6
Name: count, dtype: Int64


In [89]:
products.to_csv(CLEAN / 'products_clean.csv', index=False)

check = pd.read_csv(CLEAN / 'products_clean.csv')
print(check.shape)                                                       # expect (32951, 11)
print(check['product_id'].is_unique)                                     # True
print(check['has_size_issue'].sum())                                     # 6
print((check['product_category_name_english'] == 'unknown').sum())       # 610
print(check['product_category_name_english'].isna().sum())               # 0
print(check.columns.tolist())                                            # spelling now fixed

(32951, 11)
True
6
610
0
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'product_category_name_english', 'has_size_issue']
